<center>
<h1 style="font-family:verdana">
 💻 🧑 Classificació d'intencions (PyTorch) 🧑 💻


<p> 🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a detectar la intenció de l'usuari a partir d'interaccions reals amb un xatbot. En el context de xatbots, la classificació d'intencions ajuda a entendre quina acció o resposta hauria de prendre el sistema en funció de la consulta de l'usuari.  


<p> ✨ <b>Contingut</b>: en primer lloc, farem servir una base de dades amb oracions d'interaccions en espanyol etiquetades com 19 intencions diferents. En segon lloc, realitzarem el preprocessament de les dades, és a dir, transformarem les dades perquè tinguen un format adequat per a ser introduïdes al model. I finalment, dissenyarem i entrenarem el model de classificació per detectar automàticament la intenció de les oracions.</p>  


<p> ✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent. </p>



(nota: Hem identificat que l'idioma de la base de dades l'anglés, no el castellà, i que la partició d'entrenament conté 22 classes d'intencions, incloent-hi les etiquetes compostes, no 19.)

---

<h2> Índex </h2>

1. [Inspecció del conjunt de dades](#section-one)
  * [Exercici 1](#ex-one)
2. [Preprocessament de dades](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
3. [Disseny del model i entrenament](#section-three)
  * [Exercici 4](#ex-four)
  * [Exercici 5](#ex-five)
4. [Lliurable](#section-four)
---

In [1]:
import random
import re
from collections import Counter
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

In [2]:
from pathlib import Path
if Path('./data/train.csv').is_file() and Path('./data/test.csv').is_file():
    print('Els fitxers CSV ja estan disponibles; reutilitzem les dades locals.')
else:
    !pip install gdown
    !gdown "https://drive.google.com/uc?id=1u2wzXvsuscLeFHwXcDwMDaNDy0u_99-t"
    !tar -zxf nlu_ATIS_data.tar.gz

Els fitxers CSV ja estan disponibles; reutilitzem les dades locals.


(nota: En aquesta execució, la instal·lació de gdown falla perquè l'entorn bloqueja l'accès a la xarxa i l'ordre gdown no estè disponible. La comprovació dels CSV locals és imprescindible per executar aquesta cel·la sense aquests errors quan les dades ja existeixen. Per això reutilitzem els CSV existents i reservem les ordres originals de preparació per al cas que falten.)

<h1><a name="section-one"> 1. Inspecció del conjunt de dades </a></h1>

A la carpeta `data` tenim els diferents fitxers CSV que utilitzarem per a aquesta pràctica.

En primer lloc, llegirem les dades dels fitxers CSV amb `pandas`.

In [3]:
train_data = pd.read_csv('./data/train.csv', header=None)
val_data = train_data.tail(900)
train_data = pd.read_csv('./data/train.csv', header=None, nrows=4078)
test_data = pd.read_csv('./data/test.csv', header=None)

print('Training size:', len(train_data))
print('Validation dataset size:', len(val_data))
print('Test dataset size:', len(test_data))

Training size: 4078
Validation dataset size: 900
Test dataset size: 893


Per a aquesta primera part de la pràctica ens centrarem en la primera columna dels arxius que correspon amb les **oracions** en anglès introduïdes per l'usuari. I en la tercera columna que correspon amb la **intenció** de cada oració, és a dir, cada oració tindrà una etiqueta.

Podeu executar la cel·la següent tantes vegades com vulgueu per veure instàncies d'aquest conjunt de dades.


In [4]:
random_number = random.randint(0, len(train_data)-1)

train_sentences = list(train_data[0])
train_labels = list(s.replace('"', '') for s in train_data[2])
train_labels = list(s.replace(' ', '') for s in train_labels)

print('Sentence: ', train_sentences[random_number])
print('Intent: ', train_labels[random_number])

Sentence:  show the flights from pittsburgh to san francisco again on monday
Intent:  flight


A continuació analitzarem quantes etiquetes diferents hi ha al dataset i quines són.

In [5]:
num_labels = 0
for label in set(train_labels):
  print(f'Label {num_labels}:', label.split('.')[-1])
  num_labels += 1

print(f'\nThere are a total of {num_labels} intent labels')

Label 0: flight+airfare
Label 1: ground_service
Label 2: flight
Label 3: city
Label 4: aircraft
Label 5: ground_service+ground_fare
Label 6: capacity
Label 7: ground_fare
Label 8: restriction
Label 9: aircraft+flight+flight_no
Label 10: airfare
Label 11: distance
Label 12: abbreviation
Label 13: flight_time
Label 14: airline+flight_no
Label 15: airline
Label 16: airport
Label 17: meal
Label 18: airfare+flight_time
Label 19: cheapest
Label 20: flight_no
Label 21: quantity

There are a total of 22 intent labels


<h1><a name="section-two"> 2. Preprocessament de dades </a></h1>

En primer lloc, haurem de tokenitzar les oracions. Això consisteix a convertir el text en representacions numèriques, ja que els models esperen unitats discretes.

En aquesta pràctica farem servir una tokenització senzilla en PyTorch: construirem un diccionari de vocabulari basat en la freqüència de les paraules en el conjunt d'entrenament, igual que el Tokenizer de Keras. 0 es reserva per al *padding* i 1 per a tokens desconeguts (OOV).

Vegem com queda el vocabulari.

(nota: El tokenitzador proporcionat reserva el 0 per al padding, però assigna l'ID 1 a la paraula més freqüent i descarta les paraules desconegudes. Per tant, el codi de l'enunciat no implementa el token OOV que descriu aquest paràgraf.)

In [6]:
class SimpleTokenizer:
    def __init__(self, num_words=500):
        self.num_words = num_words
        self.word_index = {}
        self.index_word = {}

    def fit_on_texts(self, texts):
        words = []
        for text in texts:
            tokens = re.findall(r"\b\w+\b|'\w+", text.lower())
            words.extend(tokens)
        counts = Counter(words)
        sorted_words = sorted(counts.keys(), key=lambda w: (-counts[w], words.index(w)))

        # Indexing starting at 1 (0 reserved for padding)
        self.word_index = {w: i + 1 for i, w in enumerate(sorted_words[:self.num_words - 1])}
        self.index_word = {i: w for w, i in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            tokens = re.findall(r"\b\w+\b|'\w+", text.lower())
            seq = [self.word_index[w] for w in tokens if w in self.word_index]
            sequences.append(seq)
        return sequences

num_words = 500
tokenizer = SimpleTokenizer(num_words=num_words)
tokenizer.fit_on_texts(train_sentences)

vocab = tokenizer.word_index
print(vocab)

{'to': 1, 'from': 2, 'flights': 3, 'the': 4, 'on': 5, 'what': 6, 'me': 7, 'flight': 8, 'boston': 9, 'show': 10, 'san': 11, 'i': 12, 'denver': 13, 'a': 14, 'francisco': 15, 'in': 16, 'and': 17, 'atlanta': 18, 'pittsburgh': 19, 'is': 20, 'dallas': 21, 'baltimore': 22, 'all': 23, 'philadelphia': 24, 'like': 25, 'are': 26, 'list': 27, 'airlines': 28, 'of': 29, 'between': 30, 'that': 31, 'washington': 32, 'leaving': 33, 'please': 34, 'pm': 35, 'morning': 36, 'would': 37, 'fly': 38, 'for': 39, 'fare': 40, 'first': 41, 'wednesday': 42, 'after': 43, 'there': 44, 'oakland': 45, "'d": 46, 'ground': 47, 'you': 48, 'does': 49, 'trip': 50, 'transportation': 51, 'class': 52, 'arriving': 53, 'cheapest': 54, 'need': 55, 'city': 56, 'round': 57, 'with': 58, 'before': 59, 'which': 60, 'available': 61, 'have': 62, 'give': 63, 'at': 64, 'fares': 65, 'american': 66, 'afternoon': 67, 'one': 68, 'want': 69, 'how': 70, 'way': 71, 'new': 72, 'dc': 73, 'nonstop': 74, 'arrive': 75, 'earliest': 76, 'york': 77, 'g

---

 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</center></a></h1>

En aquest primer exercici us demanem que donat el vocabulari anterior convertiu la llista d'oracions de la partició d'entrenament, és a dir, `train_sentences` en seqüències d'IDs.

In [7]:
#TODO
train_sequences = tokenizer.texts_to_sequences(train_sentences)
print(train_sentences[0])
print(train_sequences[0])

i want to fly from boston at 838 am and arrive in denver at 1110 in the morning
[12, 69, 1, 38, 2, 9, 64, 416, 84, 17, 75, 16, 13, 64, 494, 16, 4, 36]


Si ho heu fet correctament hauríeu d'obtenir una llista d'IDs corresponents a les paraules de l'oració.


---
A continuació haurem d'aconseguir que totes les seqüències tinguen una longitud fixa. En PyTorch podem utilitzar funcions com `torch.nn.utils.rnn.pad_sequence` o operacions de padding manuals per ajustar la longitud de les seqüències al valor màxim de `max_sequence_length`.


In [8]:
def pad_sequences_torch(sequences, maxlen, padding='post'):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) > 0:
            trunc = seq[:maxlen]
            if padding == 'post':
                padded[i, :len(trunc)] = trunc
            elif padding == 'pre':
                padded[i, -len(trunc):] = trunc
    return torch.tensor(padded, dtype=torch.long)

max_sequence_length = max(map(len, train_sequences))
train_pad_sequences = pad_sequences_torch(train_sequences, maxlen=max_sequence_length, padding='pre')
print('Padded sequence: ', train_pad_sequences[0])

Padded sequence:  tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  12,  69,   1,  38,   2,   9,  64, 416,  84,  17,  75,  16,  13,
         64, 494,  16,   4,  36])


---

 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏</center></a></h1>

Com l'ordre de les paraules sí que importa als models que utilitzarem en aquesta pràctica, és aconsellable que el *padding* estiga al final i no al principi (`padding='post'`). Afegiu els zeros al final de la seqüència.

(nota: En el model base de l'exercici 4, el global max pooling no conserva l'ordre de les paraules. Mantindrem el padding al final tal com demana l'exercici 2, per? aquesta arquitectura concreta ?s invariant a l'ordre dels tokens.)

In [9]:
#TODO
train_pad_sequences = pad_sequences_torch(train_sequences, maxlen=max_sequence_length, padding='post')
print('Padded sequence: ', train_pad_sequences[0])

Padded sequence:  tensor([ 12,  69,   1,  38,   2,   9,  64, 416,  84,  17,  75,  16,  13,  64,
        494,  16,   4,  36,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0])


---

A continuació convertirem les classes d'intencions en etiquetes numèriques utilitzant `LabelEncoder` de scikit-learn. En PyTorch, la pràctica estàndard és utilitzar índexs enter com etiquetes (ja que `nn.CrossEntropyLoss` rep els índexs de classe directament), però si necessitem un vector one-hot podem convertir-ho fàcilment amb `torch.nn.functional.one_hot`.

In [10]:
label_encoder = LabelEncoder()
train_numerical_labels = label_encoder.fit_transform(train_labels)

print(f'Original labels: {train_labels[:5]}\n')
print(f'Encoded labels: {train_numerical_labels[:5]} \n')

Original labels: ['flight', 'flight', 'flight_time', 'airfare', 'airfare']

Encoded labels: [12 12 15  3  3] 



I a continuació convertim les etiquetes a vectors one-hot o mantenim els tensors d'enters per a PyTorch.

In [11]:
num_classes = len(np.unique(train_numerical_labels))
train_numerical_labels_tensor = torch.tensor(train_numerical_labels, dtype=torch.long)
train_encoded_labels = torch.nn.functional.one_hot(train_numerical_labels_tensor, num_classes=num_classes).float()

print('Example: \n')
print(f'Original label: {train_labels[0]}\n')
print(f'Numerical label: {train_numerical_labels[0]}\n')
print(f'One-hot: {train_encoded_labels[0]}\n')

Example: 

Original label: flight

Numerical label: 12

One-hot: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])



---

 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏</center></a></h1>

Amb la partició de validació i test haurem de realitzar els mateixos passos. Per tant, en aquest exercici us demanem que obtingueu `val_pad_sequences`, `val_encoded_labels`, `test_pad_sequences` i `test_encoded_labels`.

In [12]:
#TODO
val_sentences = list(val_data[0])
test_sentences = list(test_data[0])
val_sequences = tokenizer.texts_to_sequences(val_sentences)
test_sequences = tokenizer.texts_to_sequences(test_sentences)
val_pad_sequences = pad_sequences_torch(val_sequences, maxlen=max_sequence_length, padding='post')
test_pad_sequences = pad_sequences_torch(test_sequences, maxlen=max_sequence_length, padding='post')

In [13]:
val_labels = list(val_data[2])
test_labels = list(test_data[2])

# Clean quotation marks and whitespace from val and test labels
val_labels = [s.replace('"', '').replace(' ', '') for s in val_labels]
test_labels = [s.replace('"', '').replace(' ', '') for s in test_labels]

def remove_values_and_indices(input_list, values_to_remove, other_tensor):
    indices_to_remove = [idx for idx, item in enumerate(input_list) if item in values_to_remove]
    cleaned_list = [item for item in input_list if item not in values_to_remove]
    keep_indices = [idx for idx in range(len(input_list)) if idx not in indices_to_remove]
    cleaned_other_tensor = other_tensor[keep_indices]
    # return cleaned_list, cleaned_other_tensor
    return cleaned_list, cleaned_other_tensor, keep_indices

values_to_remove = ['day_name','airfare+flight','flight+airline','flight_no+airline']

# val_labels, val_pad_sequences = remove_values_and_indices(val_labels, values_to_remove, val_pad_sequences)
val_labels, val_pad_sequences, val_keep_indices = remove_values_and_indices(val_labels, values_to_remove, val_pad_sequences)
val_sentences = [val_data[0].iloc[i] for i in val_keep_indices]

# test_labels, test_pad_sequences = remove_values_and_indices(test_labels, values_to_remove, test_pad_sequences)
test_labels, test_pad_sequences, test_keep_indices = remove_values_and_indices(test_labels, values_to_remove, test_pad_sequences)
test_sentences = [test_data[0].iloc[i] for i in test_keep_indices]


In [14]:
# Now encode the labels
val_num_labels = label_encoder.transform(val_labels)
test_num_labels = label_encoder.transform(test_labels)

# val_encoded_labels = torch.nn.functional.one_hot(torch.tensor(val_num_labels), num_classes=num_classes).float()
val_encoded_labels = torch.nn.functional.one_hot(torch.tensor(val_num_labels, dtype=torch.long), num_classes=num_classes).float()

# test_encoded_labels = torch.nn.functional.one_hot(torch.tensor(test_num_labels), num_classes=num_classes).float()
test_encoded_labels = torch.nn.functional.one_hot(torch.tensor(test_num_labels, dtype=torch.long), num_classes=num_classes).float()


---

<h1><a name="section-three"> 3. Disseny del model i entrenament </a></h1>

En primer lloc, anem a comprovar si hi ha GPUs disponibles utilitzant PyTorch.

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU is available! Using: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. The model will be trained on CPU.")

GPU is not available. The model will be trained on CPU.


---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏</center></a></h1>

En aquest exercici haureu de dissenyar l'arquitectura del model en PyTorch. El nostre model tindrà quatre capes:

1. La primera capa serà un **embedding** (`nn.Embedding`). Assignarà a cada índex de cada paraula un vector dens de mida *embedding_dim*.
2. La segona capa serà un **global max pooling** layer. Prendrà el valor màxim al llarg de la dimensió de la seqüència, transformant l'entrada de (batch_size, sequence_length, embedding_dim) a (batch_size, embedding_dim).
3. La tercera capa serà una capa **densa** (`nn.Linear`) amb activació ReLU (`nn.ReLU`).
4. L'última capa serà una capa **densa** final (`nn.Linear`) per a produir els logits de les classes (en PyTorch, `nn.CrossEntropyLoss` inclou automàticament la funció Softmax).

📢 Podeu consultar la documentació de PyTorch [aquí](https://pytorch.org/docs/stable/nn.html).


In [16]:
#TODO
class IntentClassifierPyTorch(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(IntentClassifierPyTorch, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch_size, sequence_length)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
    
        # Apply masking to ignore padding tokens
        # pooled, _ = torch.max(embedded, dim=1)
        masked_embeddings = embedded.masked_fill(x.eq(0).unsqueeze(-1), float('-inf'))
        pooled, _ = torch.max(masked_embeddings, dim=1)  # (batch_size, embedding_dim)
        pooled = torch.where(x.ne(0).any(dim=1, keepdim=True), pooled, torch.zeros_like(pooled))

        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.fc2(out)
        return out


# Training setup
embedding_dim = 64
hidden_dim = 64
vocab_size = num_words + 1
batch_size = 32
epochs = 2

train_dataset = TensorDataset(train_pad_sequences, torch.tensor(train_numerical_labels, dtype=torch.long))
val_dataset = TensorDataset(val_pad_sequences, torch.tensor(val_num_labels, dtype=torch.long))
test_dataset = TensorDataset(test_pad_sequences, torch.tensor(test_num_labels, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

model = IntentClassifierPyTorch(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    #______________
    # for batch_x, batch_y in test_loader:
    for batch_x, batch_y in val_loader:
    #______________
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

#______________
# print(f"Test accuracy: {correct / total:.2f}")
print(f"Validation accuracy: {correct / total:.2f}")
#______________


Epoch 1/2, Loss: 1.2400
Epoch 2/2, Loss: 0.9083
Validation accuracy: 0.76


(nota: Per reservar el conjunt test fins a la selecció final de l'exercici 5, és imprescindible que executar el notebook no torne a mostrar mètriques de test durant la preparació dels experiments. Per això hem substituït únicament el loader, les frases i les etiquetes de test pels de validació en l'avaluació i la visualització de l'exercici 4. Les línies originals es conserven comentades entre delimitadors. No s'ha canviat l'arquitectura ni l'entrenament de l'exercici 4.)


Podeu veure a continuació les oracions que el model ha classificat incorrectament.

In [17]:
model.eval()
all_preds = []
with torch.no_grad():
    #______________
    # for batch_x, _ in test_loader:
    for batch_x, _ in val_loader:
    #______________
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

predicted_labels = label_encoder.inverse_transform(all_preds)

for i in range(0, len(predicted_labels)):
    #______________
    # if test_labels[i] != predicted_labels[i]:
    if val_labels[i] != predicted_labels[i]:
    #______________
        #______________
        # print('Sentence: ', test_sentences[i])
        print('Sentence: ', val_sentences[i])
        #______________
        #______________
        # print('Original label: ', test_labels[i])
        print('Original label: ', val_labels[i])
        #______________
        print('Predicted label: ', predicted_labels[i])
        print()


Sentence:  what kind of plane is used on the earliest flight from boston to san francisco afternoon
Original label:  aircraft
Predicted label:  flight

Sentence:  is fare code b the same as business class
Original label:  abbreviation
Predicted label:  flight

Sentence:  what is the cheapest ticket from baltimore to san francisco on friday august thirtieth
Original label:  airfare
Predicted label:  flight

Sentence:  what time are flights from denver to san francisco on continental airlines
Original label:  flight_time
Predicted label:  flight

Sentence:  how long is a trip from philadelphia airport to downtown philadelphia
Original label:  distance
Predicted label:  flight

Sentence:  how many seats in a 734
Original label:  capacity
Predicted label:  ground_service

Sentence:  show me the cheapest fares from san diego to miami one way
Original label:  airfare
Predicted label:  flight

Sentence:  is there a limousine service available from the pittsburgh airport
Original label:  groun

---

 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏ </center></a></h1>

Modifiqueu els següents paràmetres del model anterior i analitzeu com afecten a la seva *accuracy*:

 1. **Preprocessament.** Modifiqueu el Tokenizer per canviar la mida del vostre vocabulari i afegiu nous passos de preprocessament. Alguns possibles canvis són canviar la mida del vocabulari, treure la capitalització o fer servir *lemmatització* o *stemming*.

 2. **Mida dels Embeddings.** Proveu diferents mides d'*Embeddings* i observeu com canvia l'*accuracy* del model. Heu d'explicar les vostres conclusions.

 3. **Xarxes Convolucionals.** Afegiu capes convolucionals al vostre model (`nn.Conv1d`). Expliqueu amb detall els valors que heu provat i la vostra motivació a l'hora d'escollir-los. Recordeu, que també podeu provar diferents configuracions de *pooling* (`nn.MaxPool1d`, `nn.AdaptiveAvgPool1d`).

 4. **Xarxes Recurrents.** Afegiu capes recurrents al vostre model (`nn.LSTM`, `nn.GRU`). Expliqueu amb detall els valors que heu provat i la vostra motivació.

 5. **Regularització.** Quan proveu configuracions amb més paràmetres veureu que el model comença a tenir *overfitting* molt prompte durant l'entrenament. Afegiu *Dropout* (`nn.Dropout`) al vostre model. Heu d'explicar la vostra decisió de valors i de posició dins de la xarxa.

 6. **Balancejat de les classes.** Si analitzeu el dataset, veureu que la freqüència de les classes està molt desbalancejada. En PyTorch podeu afegir un pes per a cada classe utilitzant el paràmetre `weight` a `nn.CrossEntropyLoss(weight=class_weights)`. Calculeu un pes per a cada classe i afegiu-lo durant l'entrenament.

 ---

## Implementació sistemàtica de l'exercici 5

Implementem els tickets 1–13 amb funcions comunes i comparacions d'un sol factor. Els exercicis 1–4 es mantenen, excepte el canvi imprescindible de test a validació explicat en la nota anterior. El fitxer existent és `P2_1_alumnes.ipynb`; no existeix `P2_1_alumnes_pytorch.ipynb` en aquest projecte.

El nou pipeline de l'exercici 5 implementa PAD=0 i UNK=1 sense modificar el tokenitzador proporcionat als exercicis previs. Per aquest motiu, el seu feedforward de referència constitueix un baseline nou i coherent per a totes les comparacions del 5; no s'ha d'atribuir una diferència amb l'exercici 4 únicament a un hiperparàmetre.

**Execució segura per defecte:** només proves `smoke` amb subconjunts petits. `RUN_FULL_EX5=False` i `RUN_FINAL_TEST=False`. Els experiments definitius i el test final requereixen activar dues cel·les explícites separades. El registre detallat és a `ejercicio_5.md`.


### Tickets 1-2: protocol, configuration and preprocessing


In [18]:
# Tickets 1-2: protocol, configuration and preprocessing
import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import time
import uuid
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk.stem.snowball import SnowballStemmer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset

EX5_VERSION = '1.0-32992c0f36e6'
EX5_ROOT = Path('results/ex5')
EX5_DEDUPLICATE_VALIDATION = False  # Fixed protocol; never varied between experiments.
EX5_CACHE = {}


def ex5_canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False)


def ex5_digest(value):
    return hashlib.sha256(ex5_canonical(value).encode('utf-8')).hexdigest()


def ex5_write_json(path, value):
    """Immutable artifact: reuse identical content, reject accidental replacement."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n'
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8')) != value:
            raise FileExistsError(f'Different content already exists: {path}')
        return path
    with path.open('x', encoding='utf-8') as stream:
        stream.write(text)
    return path


def ex5_clean_label(label):
    return label.replace('"', '').replace(' ', '')


def ex5_audit(train_frame, val_frame, test_frame, excluded_labels):
    # Column positions and source-row indices are those inspected in the notebook.
    frames = {}
    for name, frame in [('train', train_frame), ('val', val_frame), ('test', test_frame)]:
        assert 0 in frame.columns and 2 in frame.columns
        prefix = 'data/test.csv' if name == 'test' else 'data/train.csv'
        frames[name] = pd.DataFrame({
            'text': frame[0].tolist(),
            'label': [ex5_clean_label(label) for label in frame[2]],
            'source_id': [f'{prefix}:{int(i)}' for i in frame.index],
        })
    assert not (set(frames['train'].source_id) & set(frames['val'].source_id))
    assert frames['train'].source_id.is_unique and frames['val'].source_id.is_unique
    assert all(isinstance(text, str) for frame in frames.values() for text in frame.text)
    classes = tuple(sorted(frames['train'].label.unique()))
    assert list(classes) == label_encoder.classes_.tolist(), 'LabelEncoder must be fitted on train only.'
    assert len(classes) == num_classes
    overlapping_texts = set(frames['train'].text) & set(frames['val'].text)
    duplicate_val_mask = frames['val'].text.isin(set(frames['train'].text))
    audit = {
        'version': EX5_VERSION,
        'source_rows_disjoint': True,
        'train_val_shared_distinct_texts': len(overlapping_texts),
        'validation_rows_with_train_text': int(duplicate_val_mask.sum()),
        'train_test_shared_distinct_texts': len(set(frames['train'].text) & set(frames['test'].text)),
        'deduplicate_validation': EX5_DEDUPLICATE_VALIDATION,
        'excluded_label_policy': list(excluded_labels),
        'classes': list(classes),
        'raw_counts': {name: len(frame) for name, frame in frames.items()},
        'raw_distributions': {name: frame.label.value_counts().sort_index().to_dict() for name, frame in frames.items()},
        'uppercase_sentences': {name: int(sum(text != text.lower() for text in frame.text)) for name, frame in frames.items()},
        'empty_sentences': {name: int(sum(not text.strip() for text in frame.text)) for name, frame in frames.items()},
    }
    if EX5_DEDUPLICATE_VALIDATION:
        frames['val'] = frames['val'].loc[~duplicate_val_mask].reset_index(drop=True)
    removed = {}
    for name in ['val', 'test']:
        frame = frames[name]
        keep = ~frame.label.isin(excluded_labels)
        removed[name] = frame.loc[~keep, 'label'].value_counts().sort_index().to_dict()
        frames[name] = frame.loc[keep].reset_index(drop=True)
        unknown = set(frames[name].label) - set(classes)
        if unknown:
            raise ValueError(f'Unseen labels outside the practice exclusion policy: {unknown}')
    audit['excluded_counts'] = removed
    audit['retained_counts'] = {name: len(frame) for name, frame in frames.items()}
    audit['retained_source_ids'] = {name: frame.source_id.tolist() for name, frame in frames.items()}
    audit['retained_distributions'] = {name: frame.label.value_counts().sort_index().to_dict() for name, frame in frames.items()}
    audit['missing_train_classes'] = {name: sorted(set(classes) - set(frame.label)) for name, frame in frames.items()}
    # Test content is NOT part of preprocessing, model identity or model selection.
    training_identity = {
        'train': frames['train'].to_dict('records'), 'val': frames['val'].to_dict('records'),
        'classes': list(classes), 'excluded_labels': list(excluded_labels),
        'deduplicate_validation': EX5_DEDUPLICATE_VALIDATION,
    }
    fingerprint = ex5_digest(training_identity)
    audit_id = ex5_digest({'dataset': fingerprint, 'implementation': EX5_VERSION})[:20]
    ex5_write_json(EX5_ROOT / 'audit' / f'{audit_id}.json', audit)
    return frames, classes, fingerprint, audit


EX5_DATA, EX5_CLASSES, EX5_DATA_FINGERPRINT, EX5_AUDIT = ex5_audit(
    train_data, val_data, test_data, values_to_remove,
)
display(pd.DataFrame(EX5_AUDIT['raw_distributions']).fillna(0).astype(int))
print('Retained examples:', EX5_AUDIT['retained_counts'])
print('Excluded labels:', EX5_AUDIT['excluded_counts'])
print('Distinct texts shared by train/validation:', EX5_AUDIT['train_val_shared_distinct_texts'])


@dataclass(frozen=True)
class Ex5Preprocessing:
    vocab_size: int = 500  # Total budget INCLUDING PAD and UNK.
    lowercase: bool = True
    stemming: bool = False
    max_sequence_length: int | None = None


@dataclass(frozen=True)
class Ex5ModelConfig:
    family: str = 'feedforward'
    embedding_dim: int = 64
    hidden_dim: int = 64
    dropout: float = 0.0
    filters: int = 64
    kernel_size: int = 3
    pooling: str = 'max'
    conv_layers: int = 1
    rnn_type: str = 'LSTM'
    bidirectional: bool = False
    num_layers: int = 1
    attention: bool = False


@dataclass(frozen=True)
class Ex5Training:
    learning_rate: float = 0.001
    batch_size: int = 32
    max_epochs: int = 25
    patience: int = 4
    min_delta: float = 0.0
    seed: int = 42
    device: str = 'auto'
    deterministic: bool = True
    cpu_threads: int = 4
    mode: str = 'full'
    smoke_train_size: int = 96
    smoke_val_size: int = 48


@dataclass(frozen=True)
class Ex5Experiment:
    name: str = 'reference'
    phase: str = 'reference'
    preprocessing: Ex5Preprocessing = field(default_factory=Ex5Preprocessing)
    model: Ex5ModelConfig = field(default_factory=Ex5ModelConfig)
    training: Ex5Training = field(default_factory=Ex5Training)
    class_weights: bool = False


class Ex5Tokenizer:
    PAD, UNK = 0, 1

    def __init__(self, config):
        if config.vocab_size < 2:
            raise ValueError('Reserve PAD=0 and UNK=1.')
        self.config = config
        self.stemmer = SnowballStemmer('english') if config.stemming else None
        self.vocabulary = {}

    def tokens(self, text):
        text = text.lower() if self.config.lowercase else text
        tokens = re.findall(r"\b\w+\b|'\w+", text)
        # Preserve stopwords and numbers. No learned transform is fitted outside train.
        return [self.stemmer.stem(token) for token in tokens] if self.stemmer else tokens

    def fit(self, texts):
        counts = Counter(token for text in texts for token in self.tokens(text))
        self.vocabulary = {word: i + 2 for i, (word, _) in enumerate(counts.most_common(self.config.vocab_size - 2))}
        observed_max = max((len(self.tokens(text)) for text in texts), default=1)
        self.max_length = self.config.max_sequence_length or max(1, observed_max)
        if self.max_length < 1:
            raise ValueError('max_sequence_length must be positive.')
        self.embedding_size = max(self.vocabulary.values(), default=1) + 1
        return self

    def transform(self, frame):
        sequences, lengths, raw_lengths, unknowns = [], [], [], 0
        for text in frame.text:
            tokens = self.tokens(text)
            raw_lengths.append(len(tokens))
            ids = [self.vocabulary.get(token, self.UNK) for token in tokens]
            unknowns += sum(token == self.UNK for token in ids)
            # Empty text has a single UNK placeholder; raw_lengths still records zero.
            ids = (ids or [self.UNK])[:self.max_length]
            lengths.append(len(ids))
            sequences.append(ids + [self.PAD] * (self.max_length - len(ids)))
        label_map = {label: i for i, label in enumerate(EX5_CLASSES)}
        return {
            'ids': torch.tensor(sequences, dtype=torch.long).reshape(len(frame), self.max_length),
            'lengths': torch.tensor(lengths, dtype=torch.long),
            'raw_lengths': torch.tensor(raw_lengths, dtype=torch.long),
            'labels': torch.tensor([label_map[label] for label in frame.label], dtype=torch.long),
            'source_ids': frame.source_id.tolist(),
            'unknown_tokens': unknowns,
            'empty_examples': sum(length == 0 for length in raw_lengths),
        }


def ex5_partition_frames(training):
    if training.mode not in ('smoke', 'full'):
        raise ValueError('mode must be smoke or full')
    frames = {name: EX5_DATA[name] for name in ['train', 'val']}
    if training.mode == 'smoke':
        # Fixed subsets independent of model/preprocessing/seed; labels stay aligned.
        frames = {
            'train': frames['train'].iloc[:training.smoke_train_size],
            'val': frames['val'].iloc[:training.smoke_val_size],
        }
    return frames


def ex5_preprocess(config):
    key = ex5_digest({'dataset': EX5_DATA_FINGERPRINT, 'preprocessing': asdict(config.preprocessing),
                      'mode': config.training.mode, 'sizes': [config.training.smoke_train_size, config.training.smoke_val_size]})
    if key not in EX5_CACHE:
        frames = ex5_partition_frames(config.training)
        tokenizer5 = Ex5Tokenizer(config.preprocessing).fit(frames['train'].text)
        parts = {name: tokenizer5.transform(frame) for name, frame in frames.items()}
        digest = hashlib.sha256()
        for name in ['train', 'val']:
            for field_name in ['ids', 'lengths', 'labels']:
                digest.update(parts[name][field_name].numpy().tobytes())
                digest.update(str(tuple(parts[name][field_name].shape)).encode())
        digest.update(str(tokenizer5.embedding_size).encode())
        EX5_CACHE[key] = {'tokenizer': tokenizer5, 'parts': parts, 'signature': digest.hexdigest()}
    return EX5_CACHE[key]


def ex5_loaders(prepared, training):
    loaders = {}
    for name, part in prepared['parts'].items():
        dataset = TensorDataset(part['ids'], part['lengths'], part['labels'])
        loaders[name] = DataLoader(
            dataset, batch_size=training.batch_size, shuffle=name == 'train', num_workers=0,
            generator=torch.Generator().manual_seed(training.seed) if name == 'train' else None,
        )
    return loaders


Retained examples: {'train': 4078, 'val': 900, 'test': 888}
Excluded labels: {'val': {}, 'test': {'airfare+flight': 1, 'day_name': 2, 'flight+airline': 1, 'flight_no+airline': 1}}
Distinct texts shared by train/validation: 93


,train,val,test
abbreviation,120,27,33
aircraft,68,13,9
aircraft+flight+flight_no,1,0,0
airfare,333,90,48
airfare+flight_time,1,0,0
airline,125,32,38
airline+flight_no,2,0,0
airport,14,6,18
capacity,13,3,21
cheapest,1,0,0


### Ticket 3: interchangeable neural architectures


In [19]:
# Ticket 3: interchangeable neural architectures
def ex5_masked_max(features, mask, dim):
    masked = features.masked_fill(~mask, float('-inf'))
    pooled = masked.max(dim=dim).values
    valid = mask.any(dim=dim)
    return torch.where(valid, pooled, torch.zeros_like(pooled))


class Ex5Feedforward(nn.Module):
    def __init__(self, config, vocab_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim, padding_idx=0)
        self.hidden = nn.Linear(config.embedding_dim, config.hidden_dim)
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.hidden_dim, num_classes)

    def forward(self, ids, lengths=None):
        features = self.embedding(ids)
        pooled = ex5_masked_max(features, ids.ne(0).unsqueeze(-1), dim=1)
        return self.classifier(self.dropout(F.relu(self.hidden(pooled))))


class Ex5CNN(nn.Module):
    def __init__(self, config, vocab_size, num_classes):
        super().__init__()
        if config.kernel_size not in (2, 3, 5) or config.conv_layers not in (1, 2):
            raise ValueError('Use kernels 2/3/5 and one or two convolutional layers.')
        if config.pooling not in ('max', 'avg'):
            raise ValueError('CNN pooling must be max or avg.')
        self.config = config
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim, padding_idx=0)
        self.convolutions = nn.ModuleList([
            nn.Conv1d(config.embedding_dim if i == 0 else config.filters, config.filters, config.kernel_size)
            for i in range(config.conv_layers)
        ])
        self.average = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.filters, num_classes)

    def forward(self, ids, lengths=None):
        lengths = ids.ne(0).sum(1) if lengths is None else lengths
        features = self.embedding(ids).transpose(1, 2)
        receptive_field = 1 + self.config.conv_layers * (self.config.kernel_size - 1)
        if features.size(2) < receptive_field:
            features = F.pad(features, (0, receptive_field - features.size(2)))
        valid_lengths = lengths
        for convolution in self.convolutions:
            features = F.relu(convolution(features))
            valid_lengths = (valid_lengths - self.config.kernel_size + 1).clamp(min=0)
            mask = torch.arange(features.size(2), device=ids.device)[None, None, :] < valid_lengths[:, None, None]
            features = features.masked_fill(~mask, 0.0)
        # Valid convolutions only: no partial windows crossing PAD/borders.
        # With no full window, the representation is zero (finite classifier biases).
        if self.config.pooling == 'max':
            pooled = ex5_masked_max(features, mask, dim=2)
        else:
            # Adaptive average includes zeros; renormalize by the valid window count.
            # Never introduce -inf into values that will be averaged.
            pooled = self.average(features).squeeze(2)
            pooled = pooled * (features.size(2) / valid_lengths.clamp(min=1)).unsqueeze(1)
        return self.classifier(self.dropout(pooled))


class Ex5Recurrent(nn.Module):
    def __init__(self, config, vocab_size, num_classes):
        super().__init__()
        if config.rnn_type not in ('LSTM', 'GRU'):
            raise ValueError('rnn_type must be LSTM or GRU')
        self.config = config
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim, padding_idx=0)
        rnn_class = nn.LSTM if config.rnn_type == 'LSTM' else nn.GRU
        self.rnn = rnn_class(config.embedding_dim, config.hidden_dim, num_layers=config.num_layers,
                             bidirectional=config.bidirectional, batch_first=True)
        dimensions = config.hidden_dim * (2 if config.bidirectional else 1)
        self.attention_score = nn.Linear(dimensions, 1) if config.attention else None
        self.dropout = nn.Dropout(config.dropout)  # Acts also with ONE recurrent layer.
        self.classifier = nn.Linear(dimensions, num_classes)

    def forward(self, ids, lengths=None):
        lengths = ids.ne(0).sum(1) if lengths is None else lengths
        safe_lengths = lengths.clamp(min=1).cpu()
        packed = pack_padded_sequence(self.embedding(ids), safe_lengths, batch_first=True, enforce_sorted=False)
        packed_output, state = self.rnn(packed)
        hidden = state[0] if self.config.rnn_type == 'LSTM' else state
        if self.attention_score is not None:
            outputs, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=ids.size(1))
            mask = torch.arange(ids.size(1), device=ids.device)[None, :] < safe_lengths.to(ids.device)[:, None]
            scores = self.attention_score(outputs).squeeze(-1).masked_fill(~mask, float('-inf'))
            weights = torch.softmax(scores, dim=1)
            pooled = (outputs * weights.unsqueeze(-1)).sum(dim=1)
        else:
            directions = 2 if self.config.bidirectional else 1
            # h_n: layers*directions, batch, hidden; concatenate BOTH final directions.
            pooled = hidden.reshape(self.config.num_layers, directions, ids.size(0), self.config.hidden_dim)[-1]
            pooled = pooled.transpose(0, 1).reshape(ids.size(0), -1)
        pooled = torch.where(lengths.to(ids.device)[:, None] > 0, pooled, torch.zeros_like(pooled))
        return self.classifier(self.dropout(pooled))


def build_model(config, vocab_size, num_classes):
    if not 0 <= config.dropout < 1:
        raise ValueError('dropout must be in [0, 1).')
    families = {'feedforward': Ex5Feedforward, 'cnn': Ex5CNN, 'rnn': Ex5Recurrent}
    if config.family not in families:
        raise ValueError(f'Unknown neural family: {config.family}')
    return families[config.family](config, vocab_size, num_classes)


### Ticket 4: common training and evaluation


In [20]:
# Ticket 4: common training and evaluation
def ex5_environment():
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'versions': {name: importlib.metadata.version(name) for name in ['torch', 'numpy', 'pandas', 'scikit-learn', 'nltk']},
            'cuda_available': torch.cuda.is_available(), 'cuda_version': torch.version.cuda,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def ex5_set_seed(training):
    import random
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(training.seed)
    np.random.seed(training.seed)
    torch.manual_seed(training.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(training.seed)
    torch.backends.cudnn.deterministic = training.deterministic
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(training.deterministic)
    torch.set_num_threads(training.cpu_threads)


def ex5_device(training):
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu') if training.device == 'auto' else torch.device(training.device)


def ex5_class_weight_table():
    counts = np.asarray([(EX5_DATA['train'].label == label).sum() for label in EX5_CLASSES], dtype=np.int64)
    assert (counts > 0).all()
    weights = counts.sum() / (len(EX5_CLASSES) * counts)
    return pd.DataFrame({'class_id': np.arange(len(counts)), 'label': EX5_CLASSES,
                         'count': counts, 'frequency': counts / counts.sum(), 'weight': weights})


def ex5_loss_components(logits, labels, class_weights=None):
    # Weighted mean CE divides SUM losses by SUM TARGET WEIGHTS, not batch size.
    numerator = F.cross_entropy(logits, labels, weight=class_weights, reduction='sum')
    denominator = labels.numel() if class_weights is None else class_weights[labels].sum()
    return numerator, denominator


def evaluate_model(model, loader, device, class_weights=None):
    model.eval()
    numerator, denominator = 0.0, 0.0
    targets, predictions = [], []
    with torch.inference_mode():
        for ids, lengths, labels in loader:
            ids, lengths, labels = ids.to(device), lengths.to(device), labels.to(device)
            logits = model(ids, lengths)
            loss_sum, normalization = ex5_loss_components(logits, labels, class_weights)
            numerator += float(loss_sum)
            denominator += float(normalization)
            targets.extend(labels.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    if denominator <= 0:
        raise ValueError('Evaluation requires examples with positive loss weight.')
    return {'loss': numerator / denominator,
            'accuracy': float(accuracy_score(targets, predictions)),
            'macro_f1': float(f1_score(targets, predictions, labels=np.arange(len(EX5_CLASSES)), average='macro', zero_division=0)),
            'recall_per_class': recall_score(targets, predictions, labels=np.arange(len(EX5_CLASSES)), average=None, zero_division=0).tolist(),
            'confusion_matrix': confusion_matrix(targets, predictions, labels=np.arange(len(EX5_CLASSES))).tolist(),
            'targets': targets, 'predictions': predictions}


def train_model(model, train_loader, val_loader, config):
    training = config.training
    if training.max_epochs < 1 or training.patience < 1:
        raise ValueError('max_epochs and patience must be positive.')
    device = ex5_device(training)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=training.learning_rate)
    weights = None
    if config.class_weights:
        weights = torch.tensor(ex5_class_weight_table().weight.tolist(), dtype=torch.float32, device=device)
    best_state, best_epoch, best_loss, stale = None, 0, float('inf'), 0
    history = []
    started = time.perf_counter()
    for epoch in range(1, training.max_epochs + 1):
        model.train()
        total_numerator, total_denominator, correct, count = 0.0, 0.0, 0, 0
        for ids, lengths, labels in train_loader:
            ids, lengths, labels = ids.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, lengths)
            numerator, denominator = ex5_loss_components(logits, labels, weights)
            loss = numerator / denominator
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss.')
            loss.backward()
            optimizer.step()
            total_numerator += float(numerator.detach())
            total_denominator += float(denominator)
            correct += int((logits.argmax(1) == labels).sum())
            count += labels.numel()
        # Unweighted validation CE for EVERY experiment, including weighted training.
        # This keeps the checkpoint criterion identical. No test loader is accepted.
        validation = evaluate_model(model, val_loader, device)
        history.append({'epoch': epoch, 'train_loss': total_numerator / total_denominator,
                        'train_accuracy': correct / count, 'validation_loss': validation['loss'],
                        'validation_accuracy': validation['accuracy'], 'validation_macro_f1': validation['macro_f1']})
        if validation['loss'] < best_loss - training.min_delta:
            best_loss, best_epoch, stale = validation['loss'], epoch, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
        if stale >= training.patience:
            break
    elapsed = time.perf_counter() - started
    assert best_state is not None
    model.load_state_dict(best_state)
    final_validation = evaluate_model(model, val_loader, device)
    assert math.isclose(final_validation['loss'], best_loss, rel_tol=1e-6, abs_tol=1e-7)
    return {'history': history, 'best_epoch': best_epoch, 'best_state': best_state,
            'validation': final_validation, 'training_seconds': elapsed}


### Ticket 5: persistent records, equivalence detection and exact resume


In [21]:
# Ticket 5: persistent records, equivalence detection and exact resume
def ex5_config_from_dict(value):
    return Ex5Experiment(name=value['name'], phase=value['phase'],
                         preprocessing=Ex5Preprocessing(**value['preprocessing']),
                         model=Ex5ModelConfig(**value['model']), training=Ex5Training(**value['training']),
                         class_weights=value['class_weights'])


def ex5_experiment_identity(config):
    return {'implementation': EX5_VERSION, 'dataset': EX5_DATA_FINGERPRINT,
            'config': asdict(config), 'environment': ex5_environment()}


def ex5_run_directory(config):
    identity = ex5_experiment_identity(config)
    experiment_id = ex5_digest(identity)[:20]
    return experiment_id, EX5_ROOT / config.training.mode / 'experiments' / experiment_id, identity


def ex5_effective_signature(config, prepared):
    return ex5_digest({'implementation': EX5_VERSION, 'dataset': EX5_DATA_FINGERPRINT,
                      'tensors': prepared['signature'], 'model': asdict(config.model),
                      'training': asdict(config.training), 'class_weights': config.class_weights,
                      'environment': ex5_environment()})


def ex5_summary_row(result):
    config, architecture, preprocessing = result['config'], result['config']['model'], result['config']['preprocessing']
    return {
        'experiment_id': result['experiment_id'], 'name': config['name'], 'phase': config['phase'],
        'family': architecture['family'],
        'architecture': architecture['rnn_type'] if architecture['family'] == 'rnn' else architecture['family'],
        'preprocessing': ex5_canonical(preprocessing), 'vocab_size': preprocessing['vocab_size'],
        'actual_embedding_size': result['actual_embedding_size'],
        'embedding_dim': architecture['embedding_dim'], 'hidden_dim': architecture['hidden_dim'],
        'kernel_size': architecture['kernel_size'] if architecture['family'] == 'cnn' else None,
        'pooling': architecture['pooling'] if architecture['family'] == 'cnn' else 'max' if architecture['family'] == 'feedforward' else 'attention' if architecture['attention'] else 'final_hidden',
        'bidirectional': architecture['bidirectional'], 'attention': architecture['attention'],
        'dropout': architecture['dropout'], 'class_weights': config['class_weights'],
        'validation_accuracy': result['validation']['accuracy'], 'validation_macro_f1': result['validation']['macro_f1'],
        'validation_loss': result['validation']['loss'], 'best_epoch': result['best_epoch'],
        'parameter_count': result['parameter_count'], 'training_seconds': result['training_seconds'],
        'incremental_training_seconds': result.get('incremental_training_seconds', result['training_seconds']),
        'seed': config['training']['seed'], 'mode': config['training']['mode'],
        'reused_equivalent': result.get('reused_equivalent', False),
        'source_experiment_id': result.get('source_experiment_id', result['experiment_id'])}


def ex5_load_result(experiment_id, mode='full'):
    path = EX5_ROOT / mode / 'experiments' / experiment_id / 'result.json'
    return json.loads(path.read_text(encoding='utf-8'))


def run_experiment(config):
    if config.training.mode == 'smoke' and config.training.max_epochs > 2:
        raise ValueError('Smoke tests are capped at two epochs.')
    experiment_id, directory, identity = ex5_run_directory(config)
    result_path = directory / 'result.json'
    if result_path.exists():
        ex5_write_json(directory / 'config.json', identity)
        return ex5_load_result(experiment_id, config.training.mode)
    prepared = ex5_preprocess(config)  # TRAIN and VALIDATION only.
    signature = ex5_effective_signature(config, prepared)
    directory.parent.mkdir(parents=True, exist_ok=True)
    ex5_write_json(directory / 'config.json', identity)
    # Explicit reuse for no-op changes, e.g. lowercase=False on already lowercase text.
    equivalent = None
    for candidate in sorted(directory.parent.glob('*/result.json')):
        previous = json.loads(candidate.read_text(encoding='utf-8'))
        if previous['effective_signature'] == signature:
            equivalent = previous
            break
    if equivalent is not None:
        result = copy.deepcopy(equivalent)
        result.update(experiment_id=experiment_id, config=asdict(config), reused_equivalent=True,
                      source_experiment_id=equivalent.get('source_experiment_id', equivalent['experiment_id']), incremental_training_seconds=0.0)
        ex5_write_json(directory / 'result.json', result)
        pd.DataFrame([ex5_summary_row(result)]).to_csv(directory / 'summary.csv', index=False)
        return result
    ex5_set_seed(config.training)
    loaders = ex5_loaders(prepared, config.training)
    model5 = build_model(config.model, prepared['tokenizer'].embedding_size, len(EX5_CLASSES))
    parameter_count = sum(parameter.numel() for parameter in model5.parameters() if parameter.requires_grad)
    attempt = directory / f'attempt-{uuid.uuid4().hex[:12]}'
    attempt.mkdir(parents=True, exist_ok=False)
    try:
        trained = train_model(model5, loaders['train'], loaders['val'], config)
        checkpoint = {'state_dict': trained['best_state'], 'config': asdict(config), 'classes': list(EX5_CLASSES),
                      'vocabulary': prepared['tokenizer'].vocabulary, 'max_length': prepared['tokenizer'].max_length,
                      'embedding_size': prepared['tokenizer'].embedding_size, 'dataset_fingerprint': EX5_DATA_FINGERPRINT,
                      'best_epoch': trained['best_epoch'], 'implementation': EX5_VERSION}
        checkpoint_path = attempt / 'best.pt'
        with checkpoint_path.open('xb') as stream:
            torch.save(checkpoint, stream)
        history_path = attempt / 'history.csv'
        pd.DataFrame(trained['history']).to_csv(history_path, index=False)
        validation = {key: value for key, value in trained['validation'].items() if key not in ('targets', 'predictions')}
        result = {'experiment_id': experiment_id, 'config': asdict(config), 'status': 'completed', 'implementation': EX5_VERSION,
                  'effective_signature': signature, 'reused_equivalent': False,
                  'actual_embedding_size': prepared['tokenizer'].embedding_size,
                  'validation': validation, 'best_epoch': trained['best_epoch'], 'parameter_count': parameter_count,
                  'training_seconds': trained['training_seconds'], 'checkpoint': checkpoint_path.as_posix(),
                  'history': history_path.as_posix(), 'environment': ex5_environment(), 'dataset_fingerprint': EX5_DATA_FINGERPRINT}
        pd.DataFrame([ex5_summary_row(result)]).to_csv(attempt / 'summary.csv', index=False)
        ex5_write_json(directory / 'result.json', result)
        return result
    except Exception as error:
        ex5_write_json(attempt / 'failure.json', {'error': type(error).__name__, 'message': str(error)})
        raise
    finally:
        del model5
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_experiment_group(experiment_configs):
    return pd.DataFrame([ex5_summary_row(run_experiment(config)) for config in experiment_configs])


def ex5_all_results(mode='full'):
    records = []
    for path in sorted((EX5_ROOT / mode / 'experiments').glob('*/result.json')):
        result = json.loads(path.read_text(encoding='utf-8'))
        if result['dataset_fingerprint'] == EX5_DATA_FINGERPRINT and result.get('implementation') == EX5_VERSION and result['environment'] == ex5_environment():
            records.append(ex5_summary_row(result))
    return pd.DataFrame(records)


def ex5_best(table):
    if table.empty or set(table['mode']) != {'full'}:
        raise ValueError('Selection requires FULL-mode validation results, never smoke or test.')
    return table.sort_values(['validation_accuracy', 'validation_macro_f1', 'validation_loss', 'reused_equivalent', 'experiment_id'],
                             ascending=[False, False, True, True, True]).iloc[0]


def ex5_config_for_row(row):
    return ex5_config_from_dict(ex5_load_result(row['experiment_id'], row['mode'])['config'])


### Tickets 6-10: declarative, sequential one-factor comparisons


In [22]:
# Tickets 6-10: declarative, sequential one-factor comparisons
EX5_REFERENCE = Ex5Experiment()


def ex5_preprocessing_phases(reference=EX5_REFERENCE):
    return {
        'vocabulary': [replace(reference, name=f'vocab_{size}', phase='vocabulary',
                               preprocessing=replace(reference.preprocessing, vocab_size=size)) for size in [250, 500, 1000, 2000]],
        'lowercase': [replace(reference, name=f'lowercase_{enabled}', phase='lowercase',
                              preprocessing=replace(reference.preprocessing, lowercase=enabled)) for enabled in [True, False]],
        'stemming': [replace(reference, name=f'stemming_{enabled}', phase='stemming',
                             preprocessing=replace(reference.preprocessing, stemming=enabled)) for enabled in [False, True]],
        'embedding': [replace(reference, name=f'embedding_{size}', phase='embedding',
                              model=replace(reference.model, embedding_dim=size)) for size in [32, 64, 128]],
    }


EX5_PREPROCESSING_PHASES = ex5_preprocessing_phases()


def ex5_combined_candidate(observed_table):
    """OPTIONAL explicit additional experiment; never assume individual gains add."""
    best_by_phase = {phase: ex5_config_for_row(ex5_best(observed_table.loc[observed_table.phase == phase]))
                     for phase in ['vocabulary', 'lowercase', 'stemming', 'embedding']}
    preprocessing = replace(EX5_REFERENCE.preprocessing,
                            vocab_size=best_by_phase['vocabulary'].preprocessing.vocab_size,
                            lowercase=best_by_phase['lowercase'].preprocessing.lowercase,
                            stemming=best_by_phase['stemming'].preprocessing.stemming)
    return replace(EX5_REFERENCE, name='explicit_combination', phase='combined_additional', preprocessing=preprocessing,
                   model=replace(EX5_REFERENCE.model, embedding_dim=best_by_phase['embedding'].model.embedding_dim))


def ex5_cnn_configs(common):
    # Kernels 2/3/5 capture local groups of different widths; 64 filters held fixed.
    return [replace(common, name=f'cnn_k{kernel}', phase='cnn_kernel',
                    model=Ex5ModelConfig(family='cnn', embedding_dim=common.model.embedding_dim,
                                         kernel_size=kernel, filters=64, pooling='max', dropout=0.0)) for kernel in [2, 3, 5]]


def ex5_recurrent_configs(common):
    variants = [('LSTM', False, False), ('LSTM', True, False), ('GRU', False, False),
                ('GRU', True, False), ('LSTM', False, True), ('LSTM', True, True)]
    return [replace(common, name=f'{"bi" if bidirectional else ""}{kind.lower()}{"_attention" if attention else ""}', phase='recurrent',
                    model=Ex5ModelConfig(family='rnn', embedding_dim=common.model.embedding_dim, hidden_dim=128,
                                         rnn_type=kind, bidirectional=bidirectional, attention=attention,
                                         num_layers=1, dropout=0.0)) for kind, bidirectional, attention in variants]


def ex5_dropout_configs(cnn, recurrent):
    return [replace(base, name=f'{base.model.family}_dropout_{value}', phase='dropout',
                    model=replace(base.model, dropout=value)) for base in [cnn, recurrent] for value in [0.0, 0.2, 0.4]]


def ex5_weight_configs(best):
    return [replace(best, name=f'class_weights_{enabled}', phase='class_weights', class_weights=enabled) for enabled in [False, True]]


EX5_CLASS_WEIGHTS = ex5_class_weight_table()
display(EX5_CLASS_WEIGHTS)


,class_id,label,count,frequency,weight
0,0,abbreviation,120,0.029426,1.544697
1,1,aircraft,68,0.016675,2.725936
2,2,aircraft+flight+flight_no,1,0.000245,185.363636
3,3,airfare,333,0.081658,0.556648
4,4,airfare+flight_time,1,0.000245,185.363636
5,5,airline,125,0.030652,1.482909
6,6,airline+flight_no,2,0.000490,92.681818
7,7,airport,14,0.003433,13.240260
8,8,capacity,13,0.003188,14.258741
9,9,cheapest,1,0.000245,185.363636


### Tickets 11-12: evidence-based tables and figures


In [23]:
# Tickets 11-12: evidence-based tables and figures
def ex5_sort_table(table, by='validation_accuracy', ascending=False):
    if by not in table.columns:
        raise KeyError(f'Choose a column from: {list(table.columns)}')
    return table.sort_values(by, ascending=ascending).reset_index(drop=True)


def ex5_figure_path(table, name, mode):
    identity = ex5_digest({'rows': table.fillna('').to_dict('records'), 'name': name, 'version': EX5_VERSION})[:16]
    path = EX5_ROOT / mode / 'figures' / f'{name}_{identity}.png'
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def ex5_subplots(figsize):
    # Standalone exports: no GUI/Tk dependency and no global backend changes.
    figure = Figure(figsize=figsize)
    FigureCanvasAgg(figure)
    return figure, figure.subplots()


def ex5_bar_plot(table, x_column, title, name, mode='full'):
    if table.empty:
        return None  # No invented/placeholder results plotted.
    path = ex5_figure_path(table, name, mode)
    if not path.exists():
        fig, axes = ex5_subplots((max(6, len(table) * 0.75), 4))
        axes.bar(table[x_column].astype(str), table.validation_accuracy)
        axes.set(xlabel=x_column.replace('_', ' '), ylabel='Validation accuracy',
                 title=('SMOKE — ' if mode == 'smoke' else '') + title, ylim=(0, 1))
        axes.tick_params(axis='x', rotation=35)
        fig.tight_layout()
        fig.savefig(path, dpi=160)
        fig.clear()
    return path.as_posix()


def ex5_history_plot(result, mode='full'):
    history = pd.read_csv(result['history'])
    table = pd.DataFrame([ex5_summary_row(result)])
    path = ex5_figure_path(table, f"history_{result['experiment_id']}", mode)
    if not path.exists():
        fig, axes = ex5_subplots((7, 4))
        axes.plot(history.epoch, history.train_loss, label='Train CE (weighted)' if result['config']['class_weights'] else 'Train CE')
        axes.plot(history.epoch, history.validation_loss, label='Validation CE (unweighted)')
        axes.axvline(result['best_epoch'], color='gray', linestyle='--', label='Selected checkpoint')
        axes.set(xlabel='Epoch', ylabel='Loss', title=('SMOKE — ' if mode == 'smoke' else '') + result['config']['name'])
        axes.legend()
        fig.tight_layout()
        fig.savefig(path, dpi=160)
        fig.clear()
    return path.as_posix()


def ex5_export_material(table, mode='full'):
    if table.empty:
        return {'status': 'No completed experiments; no numerical conclusions.'}
    path = EX5_ROOT / mode / 'tables' / f"comparison_{ex5_digest(table.fillna('').to_dict('records'))[:16]}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        table.to_csv(path, index=False)
    plots = {}
    for phase, x_column, title in [
        ('vocabulary', 'vocab_size', 'Vocabulary size'), ('embedding', 'embedding_dim', 'Embedding dimension'),
        ('cnn_kernel', 'name', 'CNN architectures'), ('recurrent', 'name', 'Recurrent architectures'),
        ('dropout', 'name', 'Dropout comparisons'), ('class_weights', 'name', 'Class-weight comparison'),
    ]:
        selected = table.loc[table.phase == phase]
        plots[phase] = ex5_bar_plot(selected, x_column, title, phase, mode)
    # One representative curve per family, selected using validation only.
    curves = {}
    for family, group in table.groupby('family'):
        best = group.sort_values(['validation_accuracy', 'validation_macro_f1'], ascending=False).iloc[0]
        curves[family] = ex5_history_plot(ex5_load_result(best.experiment_id, mode), mode)
    for row in table.loc[table.phase == 'dropout'].itertuples():
        curves[row.name] = ex5_history_plot(ex5_load_result(row.experiment_id, mode), mode)
    weights_table = table.loc[table.phase == 'class_weights']
    weight_analysis = None
    if len(weights_table) == 2:
        recalls = [ex5_load_result(row.experiment_id, mode)['validation']['recall_per_class'] for row in weights_table.itertuples()]
        recall_frame = pd.DataFrame({'label': EX5_CLASSES, **{str(row.class_weights): values for row, values in zip(weights_table.itertuples(), recalls)}})
        recall_frame['train_count'] = ex5_class_weight_table()['count'].to_numpy()
        reference_confusion = np.asarray(ex5_load_result(weights_table.iloc[0].experiment_id, mode)['validation']['confusion_matrix'])
        recall_frame['validation_support'] = reference_confusion.sum(axis=1)
        recall_frame['recall_delta_weighted_minus_plain'] = recall_frame['True'] - recall_frame['False']
        supported = recall_frame.validation_support > 0
        minority = supported & (recall_frame.train_count <= recall_frame.train_count.median())
        frequent = supported & (recall_frame.train_count >= recall_frame.train_count.quantile(0.75))
        weight_analysis = {
            'minority_definition': 'train count at/below median, validation support > 0',
            'frequent_definition': 'train count at/above 75th percentile, validation support > 0',
            'minority_mean_recall_delta': float(recall_frame.loc[minority, 'recall_delta_weighted_minus_plain'].mean()) if minority.any() else None,
            'frequent_mean_recall_delta': float(recall_frame.loc[frequent, 'recall_delta_weighted_minus_plain'].mean()) if frequent.any() else None,
            'interpretation': 'Positive delta means a measured gain; negative means a measured loss. No inference for classes absent in validation.',
        }
        recall_path = EX5_ROOT / mode / 'tables' / f"recall_{ex5_digest(recall_frame.to_dict('records'))[:16]}.csv"
        if not recall_path.exists():
            recall_frame.to_csv(recall_path, index=False)
        fig_path = ex5_figure_path(weights_table, 'per_class_recall', mode)
        if not fig_path.exists():
            fig, axes = ex5_subplots((12, 5))
            positions = np.arange(len(EX5_CLASSES))
            axes.bar(positions - 0.2, recall_frame['False'], width=0.4, label='Unweighted')
            axes.bar(positions + 0.2, recall_frame['True'], width=0.4, label='Weighted')
            axes.set(xticks=positions, xticklabels=EX5_CLASSES, ylabel='Validation recall',
                     title=('SMOKE — ' if mode == 'smoke' else '') + 'Recall with/without class weights', ylim=(0, 1))
            axes.tick_params(axis='x', rotation=90, labelsize=8)
            axes.legend()
            fig.tight_layout()
            fig.savefig(fig_path, dpi=160)
            fig.clear()
        plots['per_class_recall'] = fig_path.as_posix()
        ex5_write_json(recall_path.with_suffix('.json'), weight_analysis)
    if mode == 'smoke':
        plots['execution_check'] = ex5_bar_plot(table, 'name', 'Small execution checks; not final comparisons', 'execution_check', mode)
    best_rows = table.sort_values(['validation_accuracy', 'validation_macro_f1'], ascending=False).groupby('family', sort=False).head(1)
    best_path = path.with_name('best_' + path.name)
    if not best_path.exists():
        best_rows.to_csv(best_path, index=False)
    report_path = path.with_suffix('.md')
    if not report_path.exists():
        text = '# ' + ('SMOKE: execution checks, not scientific comparisons' if mode == 'smoke' else 'Measured validation results') + '\n\n'
        text += f'{len(table)} configuration records; {int((~table.reused_equivalent).sum())} trained records and explicit equivalences.\n\n'
        for row in best_rows.itertuples():
            text += f'- {row.family}: {row.experiment_id}, validation accuracy {row.validation_accuracy:.6f}, macro-F1 {row.validation_macro_f1:.6f}.\n'
        if weight_analysis is not None:
            text += '\nMeasured recall deltas (weighted minus unweighted):\n'
            text += ex5_canonical(weight_analysis) + '\n'
        text += '\nThese values come from stored runs. Read class support and per-class recall before interpreting minority classes. Test was not used for this selection.\n'
        report_path.write_text(text, encoding='utf-8')
    return {'comparison_table': path.as_posix(), 'best_models': best_path.as_posix(),
            'figures': plots, 'curves': curves, 'measured_report': report_path.as_posix()}


def ex5_repeat_finalists(table):
    configs = []
    for family, group in table.groupby('family'):
        best = ex5_config_for_row(ex5_best(group))
        for seed in [42, 123, 2026]:
            configs.append(replace(best, name=f'finalist_{family}_seed_{seed}', phase='replication',
                                   training=replace(best.training, seed=seed)))
    repetitions = run_experiment_group(configs)
    aggregate = repetitions.groupby('family').agg(
        validation_accuracy_mean=('validation_accuracy', 'mean'), validation_accuracy_std=('validation_accuracy', 'std'),
        validation_macro_f1_mean=('validation_macro_f1', 'mean'), validation_macro_f1_std=('validation_macro_f1', 'std'),
        validation_loss_mean=('validation_loss', 'mean'), seeds=('seed', 'nunique')).reset_index()
    assert (aggregate.seeds == 3).all()
    return repetitions, aggregate


def ex5_freeze_selection(repetitions, aggregate):
    winning = aggregate.sort_values(['validation_accuracy_mean', 'validation_macro_f1_mean', 'validation_loss_mean', 'family'],
                                    ascending=[False, False, True, True]).iloc[0]
    # Seed 42 is specified BEFORE seeing test, never retrospectively chosen.
    chosen = repetitions.loc[(repetitions.family == winning.family) & (repetitions.seed == 42)].iloc[0]
    result = ex5_load_result(chosen.experiment_id)
    selection = {'experiment_id': chosen.experiment_id, 'checkpoint': result['checkpoint'],
                 'checkpoint_sha256': hashlib.sha256(Path(result['checkpoint']).read_bytes()).hexdigest(),
                 'dataset_fingerprint': EX5_DATA_FINGERPRINT, 'seed_policy': 'fixed seed 42 after family selection by validation mean',
                 'family': winning.family, 'replications': repetitions.fillna('').to_dict('records'), 'aggregate': aggregate.to_dict('records')}
    selection_id = ex5_digest(selection)[:20]
    ex5_write_json(EX5_ROOT / 'full' / 'selections' / f'{selection_id}.json', selection)
    return selection


def run_full_protocol(include_combination=False, include_second_convolution=False):
    """EXPLICIT opt-in; this function never evaluates test."""
    tables = [run_experiment_group(configs) for configs in EX5_PREPROCESSING_PHASES.values()]
    observed = pd.concat(tables, ignore_index=True)
    if include_combination:
        tables.append(run_experiment_group([ex5_combined_candidate(observed)]))
        observed = pd.concat(tables, ignore_index=True)
    common = ex5_config_for_row(ex5_best(observed))
    tables.append(run_experiment_group(ex5_cnn_configs(common)))
    cnn_kernel_table = tables[-1]
    best_cnn = ex5_config_for_row(ex5_best(cnn_kernel_table))
    cnn_extras = [replace(best_cnn, name='cnn_average_pooling', phase='cnn_pooling', model=replace(best_cnn.model, pooling='avg'))]
    if include_second_convolution:
        cnn_extras.append(replace(best_cnn, name='cnn_two_convolutions', phase='cnn_depth', model=replace(best_cnn.model, conv_layers=2)))
    tables.append(run_experiment_group(cnn_extras))
    tables.append(run_experiment_group(ex5_recurrent_configs(common)))
    best_rnn = ex5_config_for_row(ex5_best(tables[-1]))
    tables.append(run_experiment_group([
        replace(best_rnn, name=f'recurrent_hidden_{hidden}', phase='recurrent_hidden', model=replace(best_rnn.model, hidden_dim=hidden)) for hidden in [64, 128]
    ]))
    observed = pd.concat(tables, ignore_index=True)
    best_cnn = ex5_config_for_row(ex5_best(observed.loc[observed.family == 'cnn']))
    best_rnn = ex5_config_for_row(ex5_best(observed.loc[observed.family == 'rnn']))
    tables.append(run_experiment_group(ex5_dropout_configs(best_cnn, best_rnn)))
    observed = pd.concat(tables, ignore_index=True)
    tables.append(run_experiment_group(ex5_weight_configs(ex5_config_for_row(ex5_best(observed)))))
    observed = pd.concat(tables, ignore_index=True)
    repetitions, aggregate = ex5_repeat_finalists(observed)
    selection = ex5_freeze_selection(repetitions, aggregate)
    materials = ex5_export_material(observed)
    repetition_path = EX5_ROOT / 'full' / 'tables' / f"replications_{ex5_digest(repetitions.fillna('').to_dict('records'))[:16]}.csv"
    aggregate_path = repetition_path.with_name('aggregate_' + repetition_path.name)
    if not repetition_path.exists():
        repetitions.to_csv(repetition_path, index=False)
        aggregate.to_csv(aggregate_path, index=False)
    return {'results': observed, 'repetitions': repetitions, 'aggregate': aggregate,
            'selection': selection, 'materials': materials}


### Ticket 11: sealed, explicit and single final test evaluation


In [24]:
# Ticket 11: sealed, explicit and single final test evaluation
def ex5_load_checkpoint(path, requested_config=None):
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    assert checkpoint['dataset_fingerprint'] == EX5_DATA_FINGERPRINT
    assert checkpoint['classes'] == list(EX5_CLASSES)
    config = ex5_config_from_dict(checkpoint['config'] if requested_config is None else requested_config)
    if requested_config is None:
        tokenizer5 = Ex5Tokenizer(config.preprocessing)
        tokenizer5.vocabulary = checkpoint['vocabulary']
        tokenizer5.max_length = checkpoint['max_length']
        tokenizer5.embedding_size = checkpoint['embedding_size']
    else:
        # Equivalent inputs permit weight reuse, but the requested transformation
        # must still be retained (not silently replaced by the source tokenizer).
        tokenizer5 = ex5_preprocess(config)['tokenizer']
        assert tokenizer5.embedding_size == checkpoint['embedding_size']
        assert asdict(config.model) == checkpoint['config']['model']
    model5 = build_model(config.model, tokenizer5.embedding_size, len(EX5_CLASSES))
    model5.load_state_dict(checkpoint['state_dict'])
    return model5, tokenizer5, config


def evaluate_final_test(selection, confirm=False):
    if not confirm:
        raise ValueError('Final test requires explicit confirm=True AFTER validation selection.')
    if selection['dataset_fingerprint'] != EX5_DATA_FINGERPRINT:
        raise ValueError('Selection belongs to another dataset/protocol.')
    frozen_path = EX5_ROOT / 'full' / 'selections' / f'{ex5_digest(selection)[:20]}.json'
    if not frozen_path.exists() or json.loads(frozen_path.read_text(encoding='utf-8')) != selection:
        raise ValueError('Freeze a selection from the three-seed validation comparison first.')
    checkpoint_path = Path(selection['checkpoint'])
    checksum = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
    if checksum != selection['checkpoint_sha256']:
        raise ValueError('The frozen checkpoint has changed.')
    lock = EX5_ROOT / 'full' / 'final_test_once.json'
    selection_signature = ex5_digest(selection)
    if lock.exists():
        previous = json.loads(lock.read_text(encoding='utf-8'))
        if previous['selection_signature'] != selection_signature:
            raise RuntimeError('Test is already locked to another selection. No retrospective changes allowed.')
        if previous['status'] == 'completed':
            return previous  # Reuse recorded final metrics; DO NOT evaluate twice.
        raise RuntimeError('Final evaluation was reserved/interrupted. Inspect the record before retrying.')
    frozen_result = ex5_load_result(selection['experiment_id'])
    if frozen_result['config']['training']['mode'] != 'full' or frozen_result['config']['training']['seed'] != 42:
        raise ValueError('Only the predetermined FULL seed-42 checkpoint is allowed.')
    reservation = {'selection_signature': selection_signature, 'selection': selection, 'status': 'reserved'}
    ex5_write_json(lock, reservation)
    model5, tokenizer5, config = ex5_load_checkpoint(checkpoint_path, frozen_result['config'])
    device = ex5_device(config.training)
    model5.to(device)
    try:
        # FIRST transformation of actual test texts in the experiment pipeline.
        part = tokenizer5.transform(EX5_DATA['test'])
        loader = DataLoader(TensorDataset(part['ids'], part['lengths'], part['labels']),
                            batch_size=config.training.batch_size, shuffle=False, num_workers=0)
        metrics = evaluate_model(model5, loader, device)
        target = EX5_ROOT / 'full' / 'final_test'
        target.mkdir(parents=True, exist_ok=False)
        pd.DataFrame(metrics['confusion_matrix'], index=EX5_CLASSES, columns=EX5_CLASSES).to_csv(target / 'confusion_matrix.csv')
        pd.DataFrame({'source_id': part['source_ids'], 'true_class': metrics['targets'], 'predicted_class': metrics['predictions']}).to_csv(target / 'predictions.csv', index=False)
        fig, axes = ex5_subplots((12, 10))
        picture = axes.imshow(metrics['confusion_matrix'], interpolation='nearest', cmap='Blues')
        axes.set(xticks=np.arange(len(EX5_CLASSES)), yticks=np.arange(len(EX5_CLASSES)),
                 xticklabels=EX5_CLASSES, yticklabels=EX5_CLASSES, xlabel='Predicted class', ylabel='True class',
                 title='Final test confusion matrix — frozen seed-42 checkpoint')
        for label in axes.get_xticklabels():
            label.set_rotation(90)
            label.set_fontsize(8)
        for label in axes.get_yticklabels():
            label.set_fontsize(8)
        fig.colorbar(picture, ax=axes)
        fig.tight_layout()
        fig.savefig(target / 'confusion_matrix.png', dpi=180)
        fig.clear()
        record = {**reservation, 'status': 'completed', 'metrics': {key: value for key, value in metrics.items() if key not in ('targets', 'predictions')},
                  'examples': len(part['labels']), 'environment': ex5_environment()}
        ex5_write_json(target / 'metrics.json', record)
        # Intentional state transition of the one-time lock, not replacement of a run.
        temporary = lock.with_suffix('.pending')
        with temporary.open('x', encoding='utf-8') as stream:
            json.dump(record, stream, ensure_ascii=False, indent=2, allow_nan=False)
        temporary.replace(lock)
        return record
    finally:
        del model5
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


### Ticket 13: genuine small smoke tests, isolated from actual test


In [25]:
# Ticket 13: genuine small smoke tests, isolated from actual test
def run_smoke_tests():
    checks = []
    smoke_training = Ex5Training(mode='smoke', max_epochs=2, patience=1, device='cpu', cpu_threads=2)
    ex5_set_seed(smoke_training)
    ids = torch.tensor([[2, 3, 4, 5, 6, 7, 8], [3, 4, 0, 0, 0, 0, 0], [2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]])
    lengths = torch.tensor([7, 2, 1, 0])
    variants = [Ex5ModelConfig(embedding_dim=8, hidden_dim=8)]
    variants += [Ex5ModelConfig(family='cnn', embedding_dim=8, filters=8, kernel_size=kernel, pooling=pool)
                 for kernel in [2, 3, 5] for pool in ['max', 'avg']]
    variants += [Ex5ModelConfig(family='cnn', embedding_dim=8, filters=8, kernel_size=3, conv_layers=2)]
    variants += [Ex5ModelConfig(family='rnn', embedding_dim=8, hidden_dim=8, rnn_type=kind, bidirectional=bi, attention=attention)
                 for kind, bi, attention in [('LSTM', False, False), ('LSTM', True, False), ('GRU', False, False),
                                              ('GRU', True, False), ('LSTM', False, True), ('LSTM', True, True)]]
    for model_config in variants:
        model5 = build_model(model_config, 12, len(EX5_CLASSES))
        logits = model5(ids, lengths)
        assert logits.shape == (4, len(EX5_CLASSES)) and torch.isfinite(logits).all()
        F.cross_entropy(logits, torch.tensor([0, 1, 2, 3])).backward()
        assert all(torch.isfinite(parameter.grad).all() for parameter in model5.parameters() if parameter.grad is not None)
        assert torch.equal(model5.embedding.weight.grad[0], torch.zeros(8))
        model5.eval()
        with torch.no_grad():
            assert torch.allclose(model5(ids, lengths), model5(F.pad(ids, (0, 4)), lengths), atol=1e-6, rtol=1e-5)
    checks.append(f'{len(variants)} architectures: finite forward/backward, correct logits, PAD gradient zero and padding invariance.')
    negative = torch.tensor([[[-2., -3.], [0., 0.]]])
    assert torch.equal(ex5_masked_max(negative, torch.tensor([[[True], [False]]]), 1), torch.tensor([[-2., -3.]]))
    checks.append('Negative embeddings cannot be replaced by artificial padding maxima.')
    example_frame = pd.DataFrame({'text': ['', 'UNKNOWN_ZZZ'], 'label': [EX5_CLASSES[0]] * 2, 'source_id': ['unit:0', 'unit:1']})
    probe_tokenizer = Ex5Tokenizer(Ex5Preprocessing(vocab_size=5)).fit(['alpha beta alpha'])
    probe = probe_tokenizer.transform(example_frame)
    assert probe['raw_lengths'].tolist() == [0, 1]
    assert probe['lengths'].tolist() == [1, 1] and probe['ids'][:, 0].tolist() == [1, 1]
    assert set(probe_tokenizer.vocabulary.values()).isdisjoint({0, 1})
    vocabulary_before = dict(probe_tokenizer.vocabulary)
    probe_tokenizer.transform(example_frame)
    assert probe_tokenizer.vocabulary == vocabulary_before
    stemmed = Ex5Tokenizer(Ex5Preprocessing(stemming=True)).fit(['flights flying 838 from to not when'])
    assert stemmed.tokens('flights') == [SnowballStemmer('english').stem('flights')]
    assert all(token in stemmed.tokens('from to not when 838') for token in ['from', 'to', 'not', 'when', '838'])
    synthetic_words = ' '.join(f'word{i}' for i in range(2100))
    for budget in [250, 500, 1000, 2000]:
        bounded = Ex5Tokenizer(Ex5Preprocessing(vocab_size=budget, max_sequence_length=2)).fit([synthetic_words])
        assert bounded.embedding_size == budget and min(bounded.vocabulary.values()) == 2
        assert bounded.max_length == 2
    checks.append('Empty input, UNK=1, PAD=0, reserved indices and real lengths checked.')
    reference = Ex5Experiment(name='smoke_feedforward', phase='smoke', training=smoke_training,
                              model=Ex5ModelConfig(embedding_dim=8, hidden_dim=8))
    prepared = ex5_preprocess(reference)
    assert prepared is ex5_preprocess(reference)
    lower_off = replace(reference, name='smoke_lowercase_equivalent', preprocessing=replace(reference.preprocessing, lowercase=False))
    if EX5_AUDIT['uppercase_sentences']['train'] == 0 and EX5_AUDIT['uppercase_sentences']['val'] == 0:
        assert prepared['signature'] == ex5_preprocess(lower_off)['signature']
    assert 'test' not in prepared['parts']
    # Independent check of weighted CE aggregation over unequal batches.
    logits = torch.randn(7, len(EX5_CLASSES))
    labels = torch.tensor([0, 1, 1, 2, 3, 4, 4])
    weights = torch.tensor(ex5_class_weight_table().weight.tolist(), dtype=torch.float32)
    pieces = [ex5_loss_components(logits[:3], labels[:3], weights), ex5_loss_components(logits[3:], labels[3:], weights)]
    aggregated = sum(part[0] for part in pieces) / sum(part[1] for part in pieces)
    assert torch.allclose(aggregated, F.cross_entropy(logits, labels, weight=weights), atol=1e-6)
    assert (ex5_class_weight_table()['count'] > 0).all()
    checks.append('Weighted CE numerators/denominators and class-index ordering checked.')
    # Real metrics, not fabricated losses: a large min_delta forces early stopping
    # after the second epoch, allowing a check that epoch-1 weights are restored.
    stopping_config = replace(reference, training=replace(smoke_training, max_epochs=2, min_delta=1e6))
    actual_evaluate = globals()['evaluate_model']
    states = []

    def recording_evaluate(model5, loader, device, class_weights=None):
        states.append({key: value.detach().cpu().clone() for key, value in model5.state_dict().items()})
        return actual_evaluate(model5, loader, device, class_weights)

    try:
        globals()['evaluate_model'] = recording_evaluate
        ex5_set_seed(smoke_training)
        stopping_model = build_model(reference.model, prepared['tokenizer'].embedding_size, len(EX5_CLASSES))
        loaders = ex5_loaders(prepared, smoke_training)
        stopped = train_model(stopping_model, loaders['train'], loaders['val'], stopping_config)
        assert stopped['best_epoch'] == 1 and len(stopped['history']) == 2
        assert all(torch.equal(states[0][key], stopped['best_state'][key]) for key in states[0])
        assert all(torch.equal(states[0][key], states[-1][key]) for key in states[0])
    finally:
        globals()['evaluate_model'] = actual_evaluate
    before = {key: value.clone() for key, value in stopping_model.state_dict().items()}
    evaluate_model(stopping_model, loaders['val'], torch.device('cpu'))
    assert all(torch.equal(value, stopping_model.state_dict()[key]) for key, value in before.items())
    checks.append('Early stopping restores epoch-1 weights in a controlled real run; evaluation leaves parameters unchanged.')
    reproducible_states, reproducible_histories = [], []
    for _ in range(2):
        ex5_set_seed(smoke_training)
        repeated = build_model(reference.model, prepared['tokenizer'].embedding_size, len(EX5_CLASSES))
        repeated_loaders = ex5_loaders(prepared, smoke_training)
        trained = train_model(repeated, repeated_loaders['train'], repeated_loaders['val'], reference)
        reproducible_states.append(trained['best_state'])
        reproducible_histories.append(trained['history'])
    assert reproducible_histories[0] == reproducible_histories[1]
    assert all(torch.equal(value, reproducible_states[1][key]) for key, value in reproducible_states[0].items())
    checks.append('Two independent CPU trainings with seed 42 produce identical histories and checkpoints.')
    sealed_test = EX5_DATA['test']

    class ForbiddenTest:
        def __getattribute__(self, attribute):
            raise AssertionError('Actual test must remain inaccessible during smoke/search.')

    EX5_DATA['test'] = ForbiddenTest()
    try:
        configs = [reference,
                   replace(reference, name='smoke_cnn', model=Ex5ModelConfig(family='cnn', embedding_dim=8, filters=8, kernel_size=3)),
                   replace(reference, name='smoke_bilstm_attention', model=Ex5ModelConfig(family='rnn', embedding_dim=8, hidden_dim=8, bidirectional=True, attention=True)),
                   replace(reference, name='smoke_weighted', class_weights=True)]
        table = run_experiment_group(configs)
        first = run_experiment(reference)
        checkpoint_mtime = Path(first['checkpoint']).stat().st_mtime_ns
        actual_trainer = globals()['train_model']

        def forbidden_retraining(*args, **kwargs):
            raise AssertionError('A completed/equivalent experiment must not retrain.')

        try:
            globals()['train_model'] = forbidden_retraining
            assert run_experiment(reference) == first
            if prepared['signature'] == ex5_preprocess(lower_off)['signature']:
                equivalent = run_experiment(lower_off)
                assert equivalent['reused_equivalent']
                _, equivalent_tokenizer, equivalent_config = ex5_load_checkpoint(equivalent['checkpoint'], equivalent['config'])
                assert equivalent_config.preprocessing.lowercase is False
                assert equivalent_tokenizer.config.lowercase is False
        finally:
            globals()['train_model'] = actual_trainer
        assert Path(first['checkpoint']).stat().st_mtime_ns == checkpoint_mtime
        loaded_model, loaded_tokenizer, _ = ex5_load_checkpoint(first['checkpoint'])
        loaded_part = prepared['parts']['val']
        loaded_model.eval()
        with torch.no_grad():
            assert torch.isfinite(loaded_model(loaded_part['ids'], loaded_part['lengths'])).all()
        materials = ex5_export_material(table, mode='smoke')
        # This is a real small controlled pair: same FF setup, only weights differ.
        weight_pair = table.loc[table.family == 'feedforward'].copy()
        weight_pair['phase'] = 'class_weights'
        materials['weight_plot_checks'] = ex5_export_material(weight_pair, mode='smoke')
    finally:
        EX5_DATA['test'] = sealed_test
    for action in [lambda: ex5_best(table), lambda: evaluate_final_test(None, confirm=False)]:
        try:
            action()
        except ValueError:
            pass
        else:
            raise AssertionError('Smoke selection/final test must require explicit authorization and full results.')
    checks.append('Real smoke runs saved/reloaded/resumed, no-op configuration reused without retraining, checkpoint loaded, real plots exported, actual test sealed throughout.')
    report = {'status': 'passed', 'mode': 'smoke', 'checks': checks, 'environment': ex5_environment(),
              'architecture_variants': len(variants), 'examples': {'train': len(prepared['parts']['train']['labels']), 'val': len(prepared['parts']['val']['labels'])},
              'experiment_ids': table.experiment_id.tolist(), 'materials': materials, 'test_evaluated': False}
    ex5_write_json(EX5_ROOT / 'smoke' / f"verification_{ex5_digest(report)[:16]}.json", report)
    return table, report


### Execute only smoke checks (small subsets, at most two epochs)


In [26]:
# Execute only smoke checks (small subsets, at most two epochs)
EX5_MODE = 'smoke'
if EX5_MODE == 'smoke':
    ex5_smoke_table, ex5_smoke_report = run_smoke_tests()
    display(ex5_smoke_table)
    print('SMOKE:', ex5_smoke_report['status'])
    print('Forward/backward variants:', ex5_smoke_report['architecture_variants'])
    print('Actual test evaluated:', ex5_smoke_report['test_evaluated'])
else:
    print('Smoke skipped. Full execution still requires its explicit switch below.')


SMOKE: passed
Forward/backward variants: 14
Actual test evaluated: False


,experiment_id,name,phase,family,architecture,preprocessing,vocab_size,actual_embedding_size,embedding_dim,hidden_dim,kernel_size,pooling,bidirectional,attention,dropout,class_weights,validation_accuracy,validation_macro_f1,validation_loss,best_epoch,parameter_count,training_seconds,incremental_training_seconds,seed,mode,reused_equivalent,source_experiment_id
0,dffdf900814c64f9b571,smoke_feedforward,smoke,feedforward,feedforward,"{""lowercase"":true,""max_sequence_length"":null,""...",500,228,8,8,NaN,max,False,False,0.0,False,0.000000,0.000000,2.972429,2,2094,0.061581,0.061581,42,smoke,False,dffdf900814c64f9b571
1,886305fc93b45367d13d,smoke_cnn,smoke,cnn,cnn,"{""lowercase"":true,""max_sequence_length"":null,""...",500,228,8,64,3.0,max,False,False,0.0,False,0.145833,0.015238,2.662289,2,2222,0.083272,0.083272,42,smoke,False,886305fc93b45367d13d
2,dbfb74b0c81981982747,smoke_bilstm_attention,smoke,rnn,LSTM,"{""lowercase"":true,""max_sequence_length"":null,""...",500,228,8,8,NaN,attention,True,True,0.0,False,0.000000,0.000000,3.089755,2,3367,0.319895,0.319895,42,smoke,False,dbfb74b0c81981982747
3,a2c9c36810ebfad227b9,smoke_weighted,smoke,feedforward,feedforward,"{""lowercase"":true,""max_sequence_length"":null,""...",500,228,8,8,NaN,max,False,False,0.0,True,0.000000,0.000000,3.060693,2,2094,0.060828,0.060828,42,smoke,False,a2c9c36810ebfad227b9


### Explicit FULL launch: disabled during implementation


In [27]:
# Explicit FULL launch: disabled during implementation
RUN_FULL_EX5 = False
if RUN_FULL_EX5:
    ex5_full = run_full_protocol(include_combination=False, include_second_convolution=False)
    display(ex5_sort_table(ex5_full['results']))
    display(ex5_full['aggregate'])
else:
    print('FULL not executed. Set RUN_FULL_EX5=True only when ready for the controlled battery.')


FULL not executed. Set RUN_FULL_EX5=True only when ready for the controlled battery.


### Separate, one-time final test: disabled until selection has been frozen


In [28]:
# Separate, one-time final test: disabled until selection has been frozen
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if 'ex5_full' not in globals():
        raise RuntimeError('Run/resume the full validation protocol first.')
    ex5_final_test = evaluate_final_test(ex5_full['selection'], confirm=True)
    print(ex5_final_test['metrics'])
else:
    print('Final test reserved. No test accuracy has been calculated for exercise 5.')


Final test reserved. No test accuracy has been calculated for exercise 5.



<h1><a name="section-four"> 4. Lliurable </a></h1>

Heu d'entregar un document PDF de com a **màxim 10 pàgines** que incloga els resultats de tots els exercicis així com una explicació de cadascun dels resultats i de la modificació que heu fet. L'estructura del document és:

1. Introducció.
2. Experiments i Resultats (amb raonament).
3. Conclusions.

No cal que afegiu el vostre codi al document, podeu entregar el *notebook* juntament amb el document.

 ---

Podemos probar de añadir un transfromer, no es obligatorrio, en la siguiente parte sí.

## Metodologia i material per a l'informe de l'exercici 5

- **Protocol:** entrenament exclusiu amb train; vocabulari après només amb train; checkpoint de menor entropia creuada de validació sense ponderar en tots els models. La selecció de configuracions utilitza accuracy de validació, macro-F1 com a criteri secundari i loss com a desempat. Test només s'avalua després de congelar la selecció.
- **Particions:** filtrem files completes segons la política d'etiquetes original. L'auditoria registra els IDs de fila, les distribucions, les classes absents i els textos repetits entre particions. L'opció de deduplicar validació, si s'activa abans de començar, és un protocol fix i canvia la seva empremta; no es barreja amb resultats de la partició original.
- **Preprocessament:** pressupost de vocabulari incloent PAD/UNK, minúscules configurables, Snowball anglès opcional, sense eliminar stopwords ni números. Els textos buits es representen amb UNK i es conserva la longitud original zero. El truncament es fa al final i la longitud màxima només s'aprén amb train. La biblioteca Snowball normalitza internament les paraules; la comparació de stemming es fa amb lowercase activat.
- **CNN:** kernels 2/3/5 amb 64 filtres, finestres completes i convolució vàlida, sense finestres parcials als marges. Si no hi ha cap finestra completa, la representació és zero. L'average pooling es renormalitza per les finestres vàlides. Així, augmentar el padding no altera el resultat.
- **Recurrents:** una capa, hidden 128 en la comparació inicial, concatenació correcta dels estats finals bidireccionals i atenció escalar sobre tokens vàlids. El dropout explícit s'aplica a la representació agregada, també quan només hi ha una capa recurrent.
- **Comparacions controlades:** vocabulari i embedding es varien independentment amb el feedforward. Les millores individuals no es combinen automàticament; la combinació opcional requereix un experiment addicional. CNN i recurrents comparteixen el preprocessing i embedding seleccionats amb validació. Es compara dropout 0/0.2/0.4 i després class weights sense canviar altres factors.
- **Mètriques:** macro-F1 i recall es calculen sobre el conjunt fix de classes de train; una classe absent de validació aporta zero i no permet inferir rendiment minoritari. La loss ponderada de train divideix per la suma de pesos dels targets; la loss de validació sempre és sense pesos. La mitjana/desviació de tres llavors usa 42, 123 i 2026. Es tria la família per la mitjana de validació i s'avalua el checkpoint de llavor 42, fixada a priori.
- **Gràfics:** vocabulari/embedding mostren l'efecte d'un factor amb la resta constant; les barres d'arquitectura s'interpreten junt amb paràmetres i temps; les corbes train/validation permeten inspeccionar sobreajust; dropout i pesos requereixen comparar accuracy, macro-F1 i recall per classe. Una diferència de loss ponderada i sense ponderar no és una comparació directa del mateix objectiu.
- **Resultats reals:** la cel·la smoke mostra només comprovacions d'execució. No hi ha encara conclusions científiques ni un millor model definitiu. Quan s'execute full, les funcions generaran CSV, PNG i un resum Markdown únicament a partir dels resultats guardats. La matriu de confusió de test només apareixerà després de l'avaluació final explícita.
- **Limitacions:** la partició original conté textos idèntics entre train i validació malgrat usar files diferents; l'auditoria també registra solapament textual amb test. No s'ha d'afirmar independència textual completa si es conserva el protocol original. Algunes classes són molt rares i d'altres no apareixen en validació. CPU s'ha de provar amb smoke; GPU depén de disponibilitat i determinisme de les operacions. Les mètriques de test vistes abans d'aquesta implementació als exercicis previs són un límit a l'afirmació de test mai observat històricament, encara que ara el nou procés no el consulta durant la cerca.

**Proposta d'informe de màxim 10 pàgines:** 1 pàgina d'introducció/protocol; 2 de preprocessing/embedding; 2 d'arquitectures; 2 de regularització/balanceig; 2 de finalistes, tres llavors i test final; 1 de conclusions i limitacions. Seleccioneu gràfics representatius i eviteu repetir totes les taules del registre.
